In [1]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.proportion import proportions_ztest

# Update this path with your own path if running locally
DATA_PATH = "./cleaned_data/cleaned_combined_ride_share.csv"

df = pd.read_csv(DATA_PATH)
print("Created data frame.")

Created data frame.


In [2]:
# Mann-Whiteney U Tests
# Helper function with Bonferroni correction (alpha/2 tests, p-value * 2 tests)
def run_mwu(sample_1, sample_1_name, sample_2, sample_2_name, label):
    n1 = len(sample_1)
    n2 = len(sample_2)
    
    result = mannwhitneyu(sample_1, sample_2, alternative='less', method='asymptotic')
    
    # Bonferroni adjustments
    alpha_adj = 0.05 / 2
    p_adj = min(result.pvalue * 2, 1.0)
    # Added after seeing p-values were displaying as 0.0000E+00
    p_adj_display = "< 0.0001" if p_adj < 0.0001 else f"{p_adj:.4f}"
    p_display = "< 0.0001" if result.pvalue < 0.0001 else f"{result.pvalue:.4f}"

    med_1 = sample_1.median()
    iqr_1 = sample_1.quantile(0.75) - sample_1.quantile(0.25)
    med_2 = sample_2.median()
    iqr_2 = sample_2.quantile(0.75) - sample_2.quantile(0.25)
    
    # Rank-biserial correlation
    r_rb = (2 * result.statistic) / (n1 * n2) - 1
    abs_r = abs(r_rb)
    
    if abs_r < 0.10:
        magnitude = "negligible"
    elif abs_r < 0.30:
        magnitude = "small"
    elif abs_r < 0.50:
        magnitude = "moderate"
    else:
        magnitude = "large"

    if r_rb < 0:
        direction = "shorter than"
    elif r_rb > 0:
        direction = "longer than"
    else:
        direction = "equal to"

    print(f"----- {label} -----\n")
    print(f"{sample_1_name} sample size: {n1:,} | {sample_2_name} sample size: {n2:,}")
    print(f"{sample_1_name} Median: {med_1:.2f} min | {sample_1_name} IQR: {iqr_1:.2f} min")
    print(f"{sample_2_name} Median: {med_2:.2f} min | {sample_2_name} IQR: {iqr_2:.2f} min\n")
    print(f"U statistic: {result.statistic:,.0f}")
    print(f"Raw p-value: {p_display} | Bonferroni Adjusted p-value: {p_adj_display}")
    print(f"Significant at adjusted alpha = {alpha_adj:.4f}? {result.pvalue < alpha_adj}\n")
    print(f"Rank-biserial correlation (r): {r_rb:.4f}")
    print(f"Interpretation: {sample_1_name} durations tend to be {direction} {sample_2_name} durations with a {magnitude} practical effect.\n\n")


# Test 1: Members vs Casuals
members = df[df['member_casual'] == 'member']['ride_duration_minutes']
casuals_df = df[df['member_casual'] == 'casual']
casuals = casuals_df['ride_duration_minutes']

# Test 2: Winter Casuals vs. Non-Winter Casuals
is_winter = casuals_df['month'].isin([12, 1, 2])
winter_casuals = casuals_df[is_winter]['ride_duration_minutes']
non_winter_casuals = casuals_df[~is_winter]['ride_duration_minutes']

run_mwu(members, "Member", casuals, "Casual", "Test 1: Overall Duration (Members < Casuals)")
run_mwu(winter_casuals, "Winter Casual", non_winter_casuals, "Non-Winter Casual", "Test 2: Seasonal Casual Duration (Winter < Non-Winter)")

----- Test 1: Overall Duration (Members < Casuals) -----

Member sample size: 3,701,448 | Casual sample size: 2,056,450
Member Median: 8.68 min | Member IQR: 9.55 min
Casual Median: 11.47 min | Casual IQR: 14.58 min

U statistic: 3,067,340,611,758
Raw p-value: < 0.0001 | Bonferroni Adjusted p-value: < 0.0001
Significant at adjusted alpha = 0.0250? True

Rank-biserial correlation (r): -0.1941
Interpretation: Member durations tend to be shorter than Casual durations with a small practical effect.


----- Test 2: Seasonal Casual Duration (Winter < Non-Winter) -----

Winter Casual sample size: 92,174 | Non-Winter Casual sample size: 1,964,276
Winter Casual Median: 7.62 min | Winter Casual IQR: 8.60 min
Non-Winter Casual Median: 11.70 min | Non-Winter Casual IQR: 14.80 min

U statistic: 66,798,652,352
Raw p-value: < 0.0001 | Bonferroni Adjusted p-value: < 0.0001
Significant at adjusted alpha = 0.0250? True

Rank-biserial correlation (r): -0.2621
Interpretation: Winter Casual durations tend 

In [3]:
#Two Proportion Z Tests
# Helper function with Bonferroni correction (alpha/4 tests, p-value * 4 tests)
def run_prop_test(sample_1, sample_1_name, sample_2, sample_2_name, label):
    count_1 = sample_1.sum()
    n_1 = len(sample_1)
    count_2 = sample_2.sum()
    n_2 = len(sample_2)
    
    prop_1 = count_1 / n_1
    prop_2 = count_2 / n_2
    
    z_stat, p_raw = proportions_ztest([count_1, count_2], [n_1, n_2], alternative='larger')
    
    # Bonferroni adjustments
    alpha_adj = 0.05 / 4
    p_adj = min(p_raw * 4, 1.0)
    p_display = "< 0.0001" if p_raw < 0.0001 else f"{p_raw:.4f}"
    p_adj_display = "< 0.0001" if p_adj < 0.0001 else f"{p_adj:.4f}"
    
    # Cohen's h effect size
    cohens_h = 2 * np.arcsin(np.sqrt(prop_1)) - 2 * np.arcsin(np.sqrt(prop_2))
    abs_h = abs(cohens_h)
    
    if abs_h < 0.20:
        magnitude = "negligible"
    elif abs_h < 0.50:
        magnitude = "small"
    elif abs_h < 0.80:
        magnitude = "moderate"
    else:
        magnitude = "large"

    if cohens_h < 0:
        direction = "lower than"
    elif cohens_h > 0:
        direction = "higher than"
    else:
        direction = "equal to"
    
    print(f"----- {label} -----\n")
    print(f"Group 1 total rides: {n_1:,} | Group 2 total rides: {n_2:,}")
    print(f"{sample_1_name} proportion: {prop_1:.2%} | {sample_2_name} proportion: {prop_2:.2%}\n")
    print(f"Z-statistic: {z_stat:,.2f}")
    print(f"Raw p-value: {p_display} | Bonferroni Adjusted p-value: {p_adj_display}")
    print(f"Significant at adjusted alpha = {alpha_adj:.4f}? {p_raw < alpha_adj}\n")
    print(f"Cohen's h effect size: {cohens_h:.4f}")
    print(f"Interpretation: {sample_1_name} ratio is {direction} {sample_2_name} ratio with a {magnitude} practical effect.\n\n")


df['start_hour'] = df['started_at'].str.slice(11, 13).astype(int)

# Day came from BigQuery EXTRACT where Sunday is 1 and Saturday is 7
weekdays = [2, 3, 4, 5, 6]

members_df = df[df['member_casual'] == 'member']
casuals_df = df[df['member_casual'] == 'casual']

# Test 1: Weekday ratio (Members vs Casuals)
mem_weekday = members_df['day_of_week'].isin(weekdays)
cas_weekday = casuals_df['day_of_week'].isin(weekdays)

# Test 2: E-bike ratio (Winter/Spring Casual vs Non-Winter/Spring Casual)
is_winter_spring = casuals_df['month'].isin([12, 1, 2, 3, 4, 5])
win_spr_cas_ebike = casuals_df[is_winter_spring]['rideable_type'] == 'electric_bike'
sum_fall_cas_ebike = casuals_df[~is_winter_spring]['rideable_type'] == 'electric_bike'

# Test 3: Start 7-9 AM ratio (Members vs Casuals)
mem_rush_narrow = members_df['start_hour'].isin([7, 8])
cas_rush_narrow = casuals_df['start_hour'].isin([7, 8])

# Test 4: Start 6-10 AM ratio (Members vs Casuals)
mem_rush_wide = members_df['start_hour'].isin([6, 7, 8, 9])
cas_rush_wide = casuals_df['start_hour'].isin([6, 7, 8, 9])

run_prop_test(mem_weekday, "Member Weekday", cas_weekday, "Casual Weekday", "Test 1: Weekday Ride Ratio (Members > Casuals)")
run_prop_test(win_spr_cas_ebike, "Winter/Spring Casual E-bike", sum_fall_cas_ebike, "Summer/Fall Casual E-bike", "Test 2: Casual E-Bike Ratio (Winter/Spring > Summer/Fall)")
run_prop_test(mem_rush_narrow, "Member 7-9 AM", cas_rush_narrow, "Casual 7-9 AM", "Test 3: Morning Rush 7-9 AM Ratio (Members > Casuals)")
run_prop_test(mem_rush_wide, "Member 6-10 AM", cas_rush_wide, "Casual 6-10 AM", "Test 4: Morning Rush 6-10 AM Ratio (Members > Casuals)")

----- Test 1: Weekday Ride Ratio (Members > Casuals) -----

Group 1 total rides: 3,701,448 | Group 2 total rides: 2,056,450
Member Weekday proportion: 76.45% | Casual Weekday proportion: 62.29%

Z-statistic: 360.26
Raw p-value: < 0.0001 | Bonferroni Adjusted p-value: < 0.0001
Significant at adjusted alpha = 0.0125? True

Cohen's h effect size: 0.3091
Interpretation: Member Weekday ratio is higher than Casual Weekday ratio with a small practical effect.


----- Test 2: Casual E-Bike Ratio (Winter/Spring > Summer/Fall) -----

Group 1 total rides: 547,138 | Group 2 total rides: 1,509,312
Winter/Spring Casual E-bike proportion: 73.31% | Summer/Fall Casual E-bike proportion: 66.59%

Z-statistic: 91.54
Raw p-value: < 0.0001 | Bonferroni Adjusted p-value: < 0.0001
Significant at adjusted alpha = 0.0125? True

Cohen's h effect size: 0.1467
Interpretation: Winter/Spring Casual E-bike ratio is higher than Summer/Fall Casual E-bike ratio with a negligible practical effect.


----- Test 3: Morning